# Notes for myself

Here comes all notes of all the code that I have to look into from 14-03-2025 until now

Content
1. LDA
1. MyLDA class with convex combination adaptation
1. MyLDA class with sliding window adaptation


### LDA documentation from sklearn
scikit-learn/sklearn/discriminant_analysis.py
- https://github.com/scikit-learn/scikit-learn/blob/98ed9dc73/sklearn/discriminant_analysis.py#L249
- relevant parts: 
    - lines [1-168] (basics, cov matrix, ...)
    - lines [249-836] (class LinearDiscriminantAnalysis)

## 10 MyLDA class with convex combination adaptation

#### old code

In [ ]:
# Draft
# Start from a trained model from the day before --> update from there on in the next day

def update_lda_1 (X, y, uc):
    """First draft for updating LDA from scratch using convex combinations (binary classes)
    
    inputs:
    - X: data of dimensionality N x D
    - y: labels of dimensionality N. 
         takes values {1,2}
    - uc: update coefficient
    
    outputs: 
    - updated_w: updated weight vector 
    - updated_b: updated bias (scalar)
    """

    # Assume y contains labels of {1,2}
    X_class_1 = X[y==1]
    X_class_2 = X[y==2]

    ### Update mean ---------------------------------------------

    new_mean_1 = np.mean(X_class_1,0) 
    new_mean_2 = np.mean(X_class_2,0)

    old_mean_1 = self.mean_1
    old_mean_2 = self.mean_2

    updated_mean_1 = (1-uc)*old_mean_1 + uc * new_mean_1
    updated_mean_2 = (1-uc)*old_mean_2 + uc * new_mean_2

    ### Update cov matrix ---------------------------------------

    # Option: consider pooled cov matrix at the start already?
    new_S1 = manual_cov_matrix(X_class_1)
    new_S2 = manual_cov_matrix(X_class_2)

    old_S1 = self.S1
    old_S2 = self.S2

    updated_S1 = (1-uc)*S_1 + uc * S_1
    updated_S2 = (1-uc)*S_2 + uc * S_2

    # To be defined. regularization includes shrinkage, block-toeplitz & linear tapering
    updated_S1 = function_that_applies_regularization(updated_S1)
    updated_S2 = function_that_applies_regularization(updated_S2)

    # Create pooled cov matrix (within-class S)
    # change 0.5 to 1/5 and 4/5 or smt else (classes are not balanced)
    updated_S_w = 0.5 * (updated_S1 + updated_S2)


    # To do: store the updated mean and updated cov matrix
    # To do: add logger?

    ### Apply update on classifier coefficients -----------------

    # Update weights w and bias b
    updated_w = np.dot (np.linalg.pinv(updated_S_w) , updated_mean_2 - updated_mean_1)

    # To do: change the line that computes the bias to a more suitable one
    # this bias definition is only valid for balanced classes, but that is not the case with our data
    updated_b = -0.5 * np.dot(w.T,(updated_mean_1 + updated_mean_2))

    return updated_w,updated_b

### Math for binary, balanced classes for LDA


$\mathbf{m}_k = \frac{1}{N_k} \sum^{N_k}_{n=1}({\mathbf{x_n}})$

$\mathbf{S}_k = \frac{1}{N_k-1} \mathbf{X^T}\mathbf{X} \ \ \ $ with $\mathbf{X}_k = \sum^{N_k}_{n=1}({\mathbf{x_n}-\mathbf{m}_k})$ (i.e. $\mathbf{X}_k$ is the mean-centered data of class k)

$ \mathbf{S}_W = \frac{1}{2} (\mathbf{S_2} + \mathbf{S_1}) $

$\mathbf{w} = \mathbf{S}_W^{-1}(\mathbf{m_2} - \mathbf{m_1})$

$b = -\frac{1}{2}\mathbf{w}^T(\mathbf{m_1} + \mathbf{m_2})$

### imbalanced classes with ratio 1:5 = (target:nontarget) is this correct?

$\mathbf{m}_k = \frac{1}{N_k} \sum^{N_k}_{n=1}({\mathbf{x_n}})$

$\mathbf{S}_k = \frac{1}{N_k-1} \mathbf{X^T}\mathbf{X} \ \ \ $ with $\mathbf{X}_k = \sum^{N_k}_{n=1}({\mathbf{x_n}-\mathbf{m}_k})$ (i.e. $\mathbf{X}_k$ is the mean-centered data of class k)

$ \mathbf{S}_W = \frac{1}{6} \mathbf{S_1} + \frac{5}{6} \mathbf{S_2} $

$\mathbf{w} = \mathbf{S}_W^{-1}(\mathbf{m_2} - \mathbf{m_1})$

$b = -\mathbf{w}^T(\frac{1}{6}\mathbf{m_1} + \frac{5}{6} \mathbf{m_2})$



In [ ]:
import numpy as np
from sklearn.covariance import LedoitWolf



# import logging

# logging.basicConfig(
#     filename='test_mylda.log',
#     level=logging.DEBUG

# )

class MyLDA:
    "LDA with convex combination adaptation"

    def __init__(self):
        self.trained = False
        self.w = None
        self.b = None
        self.S1 = None # cov matrix target class
        self.S2 = None # cov matrix non target class
        self.S_w = None # within class cov matrix
        self.m1 = None
        self.m2 = None

    def manual_cov_matrix(self, X):
        """cov matrix from scratch - only for explanatory purposes. For the real cov matrix, shrinkage_cov is used instead
        
        inputs:
        X: data of dimensionality N x D
        
        Note that np.cov(X) accepts data of dimensionality D x N  !
        """
        #print("DEBUG: in manual_cov_matrix function")
        #print("DEBUG: input X = ",X)

        m = np.mean(X,0)
        #print("DEBUG: m = np.mean(X,0) = ", m)
        #print("DEBUG X.shape", X.shape)
        N = X.shape[0]
        #print("DEBUG N = X.shape[0] = ", X.shape[0])
        X_mean_centered = X - m
        #print("DEBUG X_mean_centered = X-m = ", X_mean_centered)
        cov_matrix = 1/(N-1) * (np.dot(np.transpose(X_mean_centered),(X_mean_centered)))
        #print("END DEBUGGING. computed cov matrix")

        return cov_matrix   

    def shrinkage_cov(self, X):
        ledoitwolf = LedoitWolf(assume_centered=False)
        ledoitwolf.fit(X)
        return ledoitwolf.covariance_
     
    def fit(self, X, y):
        """ fit LDA from scratch for binary classes
        
        inputs:
        - X: data of dimensionality N x D
        - y: labels of dimensionality N. 
            takes values {1,2}
        
        outputs: 
        - w: weight vector 
        - b: bias (scalar)
        """
        
        # Extract class data matrix 
        X_class_1 = X[y==1]
        X_class_2 = X[y==2]

        # Obtain the class means (take mean across samples N)
        m1 = np.mean(X_class_1,0) 
        m2 = np.mean(X_class_2,0)

        # Compute the cov matrix manually (see function manual_cov_matrix() above)
        #S1 = self.manual_cov_matrix(X_class_1)
        #S2 = self.manual_cov_matrix(X_class_2)

        S1 = self.shrinkage_cov(X_class_1)
        S2 = self.shrinkage_cov(X_class_2)
        # Compute the weight vector
        S_w = 0.5 * (S1 + S2)

        #S_w = (1/6)*S1 +(5/6)*S2

        w = np.dot ( np.linalg.pinv(S_w) , m2 - m1)
        #w = np.dot ( np.linalg.pinv(S_w) , (1/6) * m2 - (5/6) * m1)

        b = -0.5 * np.dot(w.T,(m1 + m2))
        #b = -1 * np.dot(w.T,((1/6) * m1 + (5/6) * m2))


        self.m1 = m1
        self.m2 = m2
        self.S1 = S1
        self.S2 = S2
        self.S_w = S_w
        self.w = w
        self.b = b
        self.trained = True

    def decision_function(self, X):
        assert self.trained is True, "Classifier has not been trained yet"
        distance = np.dot(X,self.w.T) + self.b
        return distance
    
    

In [36]:
#Example of Ledoit Wolf
import numpy as np
from sklearn.covariance import LedoitWolf
real_cov = np.array([[.4, .2],
                    [.2, .8]])
np.random.seed(0)
X = np.random.multivariate_normal(mean=[0, 0], cov=real_cov, size=50)
print((X).shape)
y = []
for x,z in X:
    if x+z>0:
        y.append(1)
    else:
        y.append(0)    


y=np.array(y)
cov = LedoitWolf().fit(X)
print(cov.covariance_)
# array([[0.4406..., 0.1616...],
#        [0.1616..., 0.8022...]])


# myclf = MyLDA()
# myclf.fit(X,y) 
# print("Trained LDA from scratch")
# print

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA 
clf = LDA(store_covariance=True, solver="svd")
clf.fit(X, y)
print("Sklearn's W cov matrix: \n", clf.covariance_)
print("Sklearn's w: ", clf.coef_)



(50, 2)
[[0.44067332 0.16161585]
 [0.16161585 0.80226294]]
Sklearn's W cov matrix: 
 [[ 0.25626226 -0.04009834]
 [-0.04009834  0.37656805]]
Sklearn's w:  [[3.32327796 3.89687547]]


In [ ]:
from blockmatrix import SpatioTemporalMatrix, fortran_block_levinson, fortran_cov_mean_transformation, linear_taper


'which' is not recognized as an internal or external command,
operable program or batch file.


#### Old/original

In [20]:
# Test whether my LDA works as sklearn's LDA
# Creating fake data X and y
X_train = [[3,10],
          [2,19],
          [1,10],
          [6,24],
          [1,7],
          [8,19],
          [1,8],
          [5,21],
          [6,19],
          [2,22],
          [7,8],
          [5,10]]
X_train = np.array(X_train) 

# X has dimensionality N x D
# print(X.shape) # (8,2)

# Corresponding labels (1: class 0, 2: class 1) 
# Note that the classes are of equal size
y_train = np.array([1, 2, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1])
#y_train = np.array([1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2])

# Compute output of my LDA
myclf = MyLDA()
myclf.fit(X_train,y_train) 
print("Trained LDA from scratch")
print("Obtained w vector w = \t\t", myclf.w)
print("Obtained bias b = \t\t",myclf.b)  
 

# Testing decision_function()
x = [[2, 6],
     [1,7],
     [4, 19],
     [3,8]]
x = np.array(x)
y = np.array([1,2,2,1])
print("MyLDA decision function: \t\t",myclf.decision_function(x))


print("means: \n", np.array([myclf.m1, myclf.m2]))
print("Cov matrix 1", myclf.S1)
print("Cov matrix 2", myclf.S2)
print("W cov matrix: \n", myclf.S_w)

# Compute output of sklearn's LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA 
clf = LDA(store_covariance=True, solver="svd")
clf.fit(X_train, y_train)

print("\nTrained LDA from sklearn")
print("Obtained w vector w = \t\t",clf.coef_[0])
print("Obtained bias b = \t\t", clf.intercept_[0])
print("sklearn's LDA decision function: \t",clf.decision_function(x))
print("Sklearn's means: \n", clf.means_)

print("Sklearn's W cov matrix: \n", clf.covariance_)
print("Smt ", clf.priors_)

clf = LDA(store_covariance=True, solver="lsqr",shrinkage="auto")
clf.fit(X_train, y_train)

print("\nTrained LDA from sklearn")
print("Obtained w vector w = \t\t",clf.coef_[0])
print("Obtained bias b = \t\t", clf.intercept_[0])
print("sklearn's LDA decision function: \t",clf.decision_function(x))
print("Sklearn's means: \n", clf.means_)

print("Sklearn's W cov matrix: \n", clf.covariance_)
print("Smt ", clf.priors_)

# myclf.update_lda_cc(x,y,0.005)
# print("Update LDA with convex combination")
# print("Updated w vector w = \t\t", myclf.w)
# print("Updated bias b = \t\t",myclf.b)   


Trained LDA from scratch
Obtained w vector w = 		 [-0.75420899  0.05019052]
Obtained bias b = 		 2.21367509972358
MyLDA decision function: 		 [1.00640022 1.81079972 0.15045894 0.35257226]
means: 
 [[ 5.66666667 15.33333333]
 [ 2.16666667 14.16666667]]
Cov matrix 1 [[ 4.98894642  2.25811923]
 [ 2.25811923 36.12216469]]
Cov matrix 2 [[ 4.80931788  5.51105759]
 [ 5.51105759 34.13512657]]
W cov matrix: 
 [[ 4.89913215  3.88458841]
 [ 3.88458841 35.12864563]]

Trained LDA from sklearn
Obtained w vector w = 		 [-1.20144141  0.11522377]
Obtained bias b = 		 3.0060948482961143
sklearn's LDA decision function: 	 [1.29455468 2.61121986 0.38958092 0.32356082]
Sklearn's means: 
 [[ 5.66666667 15.33333333]
 [ 2.16666667 14.16666667]]
Sklearn's W cov matrix: 
 [[ 2.84722222  4.375     ]
 [ 4.375      37.18055556]]
Smt  [0.5 0.5]

Trained LDA from sklearn
Obtained w vector w = 		 [-1.22739667 -0.02406975]
Obtained bias b = 		 5.162332465069479
sklearn's LDA decision function: 	 [ 2.56312061  3.766447

In [ ]:
# check if manual cov matrix does the same as np.cov --> Yes
X_train = [[3,20],
          [2,7],
          [1,10],
          [6,24],
          [1,7],
          [8,19],
          [1,8],
          [5,21]]
X_train = np.array(X_train) 
y_train = np.array([1, 2, 2, 1, 2, 1, 2, 1])

print(np.cov(X_train.T))
print(MyLDA().manual_cov_matrix(X_train))

x = [[3, 6],
     [1,7]]
x = np.array(x)
y = np.array([1,2])

print(np.cov(x.T))
print(MyLDA().manual_cov_matrix(x))

print(MyLDA.)


[[ 7.125      15.5       ]
 [15.5        51.14285714]]
[[ 7.125      15.5       ]
 [15.5        51.14285714]]
[[ 2.  -1. ]
 [-1.   0.5]]
[[ 2.  -1. ]
 [-1.   0.5]]
<class '__main__.MyLDA'>


In [ ]:
# Draft: Start from a trained model from the day before --> update from there on in the next day
    def update_lda_cc (self, X, y, uc):
        """First draft for updating LDA from scratch using convex combinations (binary classes)
        
        inputs:
        - X: data of dimensionality N x D. Should be an numpy array!
        - y: labels of dimensionality N. 
            takes values {1,2}
        - uc: update coefficient
        
        outputs: 
        - updated_w: updated weight vector 
        - updated_b: updated bias (scalar)
        """

        # For debugging
        print("Inside the update_lda_cc")
        # print("Inputs: ")
        # print("X: ",X)
        # print("y: ",y)
        #print("uc: ",uc)
        
        # Assume y contains labels of {1,2}
        X_class_1 = X[y==1]
        X_class_2 = X[y==2]
        # print("X_class_1: ",X_class_1)
        # print("X_class_2: ",X_class_2)

        ### Update mean ---------------------------------------------

        new_mean_1 = np.mean(X_class_1,0) 
        new_mean_2 = np.mean(X_class_2,0)
        #print("new mean 1:",new_mean_1)
        #print(new_mean_1.shape)
        #print("new mean 2:",new_mean_2)

        old_mean_1 = self.m1
        old_mean_2 = self.m2
        #print("old mean 1:",old_mean_1)
        #print("old mean 2:",old_mean_2)

        updated_mean_1 = (1-uc)*old_mean_1 + uc * new_mean_1
        updated_mean_2 = (1-uc)*old_mean_2 + uc * new_mean_2
        #print("updated mean 1:",updated_mean_1)
        #print("updated mean 2:",updated_mean_2)

        ### Update cov matrix ---------------------------------------

        # temporary solution: replace self.manual_cov_matrix by np.cov because of an error. The bug has not been fixed yet
        # try: 
        #     new_S1 = np.cov(X_class_1.T)
        #     new_S2 = np.cov(X_class_2.T)
        # except: 
        #     print("Error")
        # I commented the above out bc my manual cov matrix gives the exact same results as np.cov(X.T)
        
        # Option: consider pooled cov matrix at the start already?
        new_S1 = self.manual_cov_matrix(X_class_1)
        new_S2 = self.manual_cov_matrix(X_class_2)
        # print("new_S1: ",new_S1)
        # print("new_S2: ",new_S2)
        
        old_S1 = self.S1
        old_S2 = self.S2
        # print("old_S1: ", old_S1)
        # print("old_S2: ", old_S2)

        print("Now doing a convex combi of both new and old cov matrix...")
        updated_S1 = (1-uc)*old_S1 + uc * new_S1
        updated_S2 = (1-uc)*old_S2 + uc * new_S2
        print("updated_S1: ", updated_S1)
        print("updated_S2: ", updated_S2)
        # To be defined. regularization includes shrinkage, block-toeplitz & linear tapering
        # updated_S1 = function_that_applies_regularization(updated_S1)
        # updated_S2 = function_that_applies_regularization(updated_S2)

        # Create pooled cov matrix (within-class S)
        # change 0.5 to 1/5 and 4/5 or smt else (classes are not balanced)
        print("Now updating the pooled cov matrix (0.5 each)")
        updated_S_w = 0.5 * (updated_S1 + updated_S2)
        print("updated_S_w: ", updated_S_w)

        print("Succesful")
        #updated_S_w = (4/5) * updated_S1 + (1/5) * updated_S2 # assuming S1 is non target, S2 is target (double check this!)

        ### Apply update on classifier coefficients -----------------

        assert not np.any(np.isnan(updated_S_w)), "Covariance matrix contains NaNs"
        assert not np.any(np.isinf(updated_S_w)), "Covariance matrix contains Infs"

        # Update weights w and bias b
        print("Calculating the ivnerse of the cov matrix")
        inv_cov = np.linalg.pinv(updated_S_w) 
        print("Succesful")
        print("Doing dot product to obtain w")
        updated_w = np.dot (inv_cov, updated_mean_2 - updated_mean_1)
        print("Succesful")

        # To do: change the line that computes the bias to a more suitable one
        # this bias definition is only valid for balanced classes, but that is not the case with our data
        print("Updating the bias")
        updated_b = -0.5 * np.dot(updated_w.T,(updated_mean_1 + updated_mean_2))
        print("Succesful")

        self.w = updated_w
        self.b = updated_b
        self.S_w = updated_S_w
        self.S1 = updated_S1
        self.S2 = updated_S2
        self.m1 = updated_mean_1
        self.m2 = updated_mean_2

        return updated_w, updated_b
        
